# 33. Gensim Topics & Word Vectors — `gensim`
**Python Libraries Lab · Phase 2 · Natural Language Processing**

_Discover topics in documents and meaning in word vectors._

✅ **Runs in Colab** as-is.

Gensim is a library for finding structure in large collections of text without labels. Two ideas are at its heart. Topic models, like LDA, read many documents and discover the themes running through them. Word vectors, like Word2Vec, turn each word into a list of numbers, so that words used in similar ways end up close together.

**In this notebook:** You will build a small corpus of ShopEasy support tickets, discover their main topics with LDA, train Word2Vec on them, find similar words, and measure how similar two tickets are.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q gensim
import importlib.metadata as md
for pkg in ['gensim']:
    print(pkg, md.version(pkg))

## Step 1 · A corpus of support tickets
400 short tickets generated from five real-looking themes.

In [ ]:
import random
random.seed(0)
themes = {
    "delivery": ["delivery late courier tracking package delayed", "parcel not arrived courier tracking shipping", "delivery date missed package courier"],
    "refund": ["refund money not received bank account", "refund status pending money back", "return refund amount credited bank"],
    "quality": ["product broken poor quality damaged", "item damaged cracked quality bad", "material quality cheap broken stitching"],
    "payment": ["payment failed card upi charged twice", "upi payment deducted order failed", "card payment declined bank otp"],
    "account": ["password reset login account locked", "cannot login otp account email", "account email change password"],
}
tickets = []
for _ in range(400):
    theme = random.choice(list(themes))
    words = random.choice(themes[theme]).split() + random.choice(themes[theme]).split()
    random.shuffle(words)
    tickets.append(words[:8])
print(len(tickets), "tickets, e.g.:", " ".join(tickets[0]))

## Step 2 · Dictionary and bag of words
Each word gets an id; each ticket becomes (word id, count) pairs.

In [ ]:
from gensim import corpora
dictionary = corpora.Dictionary(tickets)
bow = [dictionary.doc2bow(t) for t in tickets]
print(len(dictionary), "unique words")
print(bow[0])
print([(dictionary[i], n) for i, n in bow[0]])

## Step 3 · Discover topics with LDA
We ask for 5 topics and see which words define each.

In [ ]:
from gensim.models import LdaModel
lda = LdaModel(bow, num_topics=5, id2word=dictionary, passes=10, random_state=0)
for i, words in lda.show_topics(num_words=5, formatted=False):
    print(f"Topic {i}:", ", ".join(w for w, _ in words))

new = dictionary.doc2bow("my upi payment failed but money deducted".split())
print("\nNew ticket topics:", [(t, round(p, 2)) for t, p in lda.get_document_topics(new)])

## Step 4 · Train Word2Vec
Each word becomes a 50-number vector learned from its neighbours.

In [ ]:
from gensim.models import Word2Vec
w2v = Word2Vec(sentences=tickets, vector_size=50, window=4, min_count=2, epochs=50, seed=0, workers=1)
print("Vector for 'refund' (first 6 numbers):", w2v.wv["refund"][:6].round(2))
for w in ["refund", "courier", "password"]:
    print(f"Most like {w!r}:", [x for x, _ in w2v.wv.most_similar(w, topn=4)])

## Step 5 · How similar are two tickets?
Average word vectors, then compare with cosine similarity (1 = same direction).

In [ ]:
import numpy as np
def ticket_vec(words):
    return np.mean([w2v.wv[w] for w in words if w in w2v.wv], axis=0)
def similarity(a, b):
    va, vb = ticket_vec(a.split()), ticket_vec(b.split())
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

print("refund vs refund:  ", round(similarity("refund not received", "money back pending refund"), 2))
print("refund vs delivery:", round(similarity("refund not received", "courier package delayed"), 2))

## Step 6 · Optional: load pre-trained vectors
Real projects use vectors trained on billions of words. This downloads about 66 MB, so it is optional; set `TRY = True` to run it.

In [ ]:
TRY = False
if TRY:
    import gensim.downloader as api
    glove = api.load("glove-wiki-gigaword-50")
    print(glove.most_similar("delivery", topn=5))
    print("king - man + woman =", glove.most_similar(positive=["king", "woman"], negative=["man"], topn=1))
else:
    print("Skipped. Set TRY = True to download GloVe vectors.")

## ✍️ Your turn
Store the **single word most similar to `'upi'`** in your `w2v` model in `answer` (just the word).

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    exp = w2v.wv.most_similar("upi", topn=1)[0][0]
    return answer == exp, f"Expected {exp!r}, got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = w2v.wv.most_similar("upi", topn=1)[0][0]

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Corpus**
- **Dictionary**
- **Bag of words**
- **LDA topic model**
- **Word2Vec**
- **Similarity**

**Next:** 34. OpenCV Computer Vision (`opencv-python`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/34_opencv.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install gensim
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.